# R02 · Priority inversion, mutexes vs semaphores, deadlock

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/05_realtime_linux/R02_priority_inversion_mutexes_vs_semaphores_deadloc.ipynb)

**Track:** Real-time and Linux · **Skill axis (sheet):** Real-time and Linux · **Time:** about 55 min

**Skill Hub concepts:** Priority inversion and inheritance; Semaphore versus mutex; Deadlock conditions

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("R02")

## Map: how a low-priority task can stall a high-priority one

```
 prio 3  H        ██ waits for M ............................██████ done          (no inheritance)
 prio 2  M-task      ███████████████████ (runs while H waits!)
 prio 1  L   ██[lock M]█                 ███████ unlock M
 time    0  1  2  3                  9            12     14
```
Mars Pathfinder (1997) reset itself this way until priority inheritance was switched on in VxWorks.

| Tool | Owner? | Priority inheritance? | Give from an ISR? | Use for |
|---|---|---|---|---|
| Mutex | yes, only the locker unlocks | yes (FreeRTOS mutexes, `PTHREAD_PRIO_INHERIT`) | no | protecting shared data |
| Binary semaphore | no | no | yes | signaling an event (ISR -> task) |
| Counting semaphore | no | no | yes | counting free slots or pending events |

**Priority inheritance:** while a task holds a mutex that a higher-priority task is waiting for, it runs at that
higher priority (transitively along chains). **Priority ceiling** raises it to the highest priority of any user up front.

**Deadlock needs all four Coffman conditions:** mutual exclusion, hold and wait, no preemption, circular wait.
Break one (most often circular wait, with a global lock order, or hold-and-wait, with timeouts).

## Task 1: effective priority with inheritance
The scheduler below is given. Write `effective_priority`: the task's base priority, or with inheritance on, the
highest effective priority of any task blocked on a mutex this task owns (follow chains).

In [ ]:
def effective_priority(task, tasks, owner, inheritance):
    raise NotImplementedError  # TODO

def simulate(spec, inheritance, t_max=100):
    """Single core, preemptive fixed priority, 1 tick granularity. lock/unlock take no time."""
    tasks = [dict(name=n, prio=p, release=r, script=list(s), pc=0, left=None, finish=None) for n, p, r, s in spec]
    owner, t, timeline = {}, 0, []
    def blocked(task):
        op = task["script"][task["pc"]]
        return op[0] == "lock" and owner.get(op[1]) not in (None, task)
    while t < t_max and any(x["finish"] is None for x in tasks):
        for x in tasks:                                  # zero-time ops for every runnable task
            while x["release"] <= t and x["finish"] is None and not blocked(x) and x["script"][x["pc"]][0] in ("lock", "unlock"):
                op = x["script"][x["pc"]]
                if op[0] == "lock":
                    owner[op[1]] = x
                else:
                    owner[op[1]] = None
                x["pc"] += 1
                if x["pc"] == len(x["script"]):
                    x["finish"] = t
        ready = [x for x in tasks if x["release"] <= t and x["finish"] is None and not blocked(x)]
        if not ready:
            timeline.append("."); t += 1; continue
        cur = max(ready, key=lambda x: (effective_priority(x, tasks, owner, inheritance), -x["release"]))
        if cur["left"] is None:
            cur["left"] = cur["script"][cur["pc"]][1]
        cur["left"] -= 1
        timeline.append(cur["name"])
        t += 1
        if cur["left"] == 0:
            cur["left"] = None
            cur["pc"] += 1
            if cur["pc"] == len(cur["script"]):
                cur["finish"] = t
    return {x["name"]: x["finish"] for x in tasks}, "".join(timeline)

PATHFINDER = [
    ("L", 1, 0, [("run", 1), ("lock", "M"), ("run", 4), ("unlock", "M"), ("run", 1)]),
    ("H", 3, 2, [("run", 1), ("lock", "M"), ("run", 2), ("unlock", "M")]),
    ("M", 2, 3, [("run", 6)]),
]
try:
    for inh in (False, True):
        fin, tl = simulate(PATHFINDER, inh)
        print(f"inheritance={inh!s:5}  timeline {tl}   finish {fin}")
except NotImplementedError:
    print("finish Task 1 first")

In [ ]:
lab.check("inheritance", effective_priority, simulate)

In [ ]:
%%writefile r02_inheritance.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/r02.h
// Grade it with lab.check("inheritance", "r02_inheritance.c"). For C++, name the file .cpp in both places.
#include "r02.h"

int effective_priority(const RtTask *tasks, size_t n, size_t task, const int *mutex_owner, bool inheritance) {
    (void)n; (void)mutex_owner; (void)inheritance;
    return tasks[task].prio; /* TODO: inherit from every task waiting on a mutex this task holds (recursively) */
}

In [ ]:
# lab.check("inheritance", "r02_inheritance.c")  # uncomment after writing the C/C++ cell above

## Task 2: find a deadlock
`waits_for` maps each task to the task holding the lock it is waiting for (or `None`). Return the set of tasks on a
cycle, or `None`.

In [ ]:
def find_cycle(waits_for):
    raise NotImplementedError  # TODO

lab.check("deadlock", find_cycle)

In [ ]:
%%writefile r02_deadlock.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/r02.h
// Grade it with lab.check("deadlock", "r02_deadlock.c"). For C++, name the file .cpp in both places.
#include "r02.h"

size_t find_cycle(const int *waits_for, size_t n, int *cycle_out) {
    (void)waits_for; (void)n; (void)cycle_out;
    return 0; /* TODO: follow the waits-for chain from each task until it ends or repeats */
}

In [ ]:
# lab.check("deadlock", "r02_deadlock.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Analyze).** In the Pathfinder scenario without inheritance, how many ticks is H blocked waiting for the mutex?

**M2 (Analyze).** Same, with inheritance?

**M3 (Remember).** How many conditions must hold at once for a deadlock?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Apply).** A UART interrupt must wake a parser task. Which primitive?
- a) a mutex the ISR unlocks
- b) a binary semaphore or task notification given from the ISR (`xSemaphoreGiveFromISR` / `vTaskNotifyGiveFromISR`)
- c) a busy-wait flag

**K2 (Evaluate).** Two tasks each lock `spi_mutex` and `can_mutex`, in opposite orders, and the robot freezes once a
day. Explain the deadlock with the Coffman conditions and give two fixes.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `r02.h`).
3. In `roboprep/checks/r02.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()